# 04 · Selected-Model Evaluation, Review Policy and Inference

The model family selected on tuning is evaluated through frozen holdout metrics, review-capacity constraints
and expected costs. All five families remain available in the comparison table. Unlabeled test transactions receive
scores and actions from the selected artifact; their labels are unavailable for predictive-performance measurement.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "fraud_detection").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from fraud_detection.config import Config
from fraud_detection.data import ID, TARGET, TIME

config = Config.load(ROOT / "configs" / "comparison.yaml")
processed_dir = Path(config.processed_dir)
artifact_dir = Path(config.output_dir)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.2})


def read_json(name):
    return json.loads((artifact_dir / name).read_text())

In [ ]:
%matplotlib inline

## 1. Holdout Model Comparison

Average precision reflects ranking under class imbalance, ROC-AUC summarizes discrimination,
and log loss and Brier score assess probability quality. Baseline comparisons establish
the improvement beyond class prevalence and a small-feature logistic model.

In [ ]:
from fraud_detection.metrics import classification_metrics
from fraud_detection.policy import ReviewState, business_metrics, daily_top_k, online_review

metrics = read_json("metrics.json")
selection = read_json("selection.json")
holdout = pd.read_parquet(artifact_dir / "holdout_predictions.parquet")
labels = holdout[TARGET].to_numpy()
scores = holdout.risk_score.to_numpy()
recomputed = classification_metrics(labels, scores)
assert np.isclose(
    recomputed["average_precision"], metrics["selected_calibrated"]["average_precision"]
)
comparison = pd.read_csv(artifact_dir / "model_comparison.csv")
selected_name = selection["model"]
display(
    comparison[
        [
            "model",
            "selected_on_tuning",
            "tuning_average_precision",
            "average_precision",
            "roc_auc",
            "brier_score",
            "log_loss",
            "review_precision",
            "review_recall",
            "expected_savings",
        ]
    ]
)
display(pd.Series({"selected_model": selected_name, "threshold": selection["threshold"]}))

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

precision, recall, _ = precision_recall_curve(labels, scores)
fpr, tpr, _ = roc_curve(labels, scores)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(recall, precision)
axes[0].axhline(labels.mean(), color="grey", linestyle="--", label="Class prior")
axes[0].set(title="Holdout precision-recall", xlabel="Recall", ylabel="Precision")
axes[0].legend()
axes[1].plot(fpr, tpr)
axes[1].plot([0, 1], [0, 1], color="grey", linestyle="--")
axes[1].set(title="Holdout ROC", xlabel="False positive rate", ylabel="True positive rate")
plt.tight_layout()
plt.show()

## 2. Online Review Allocation

After n transactions within a relative day, cumulative reviews cannot exceed floor(0.01 × n).
Events follow (TransactionDT, TransactionID) order and require a score above the frozen threshold.
This policy enforces capacity without future daily volume, although it may miss early fraud or leave capacity unused.

In [ ]:
reviewed = online_review(holdout, scores, selection["threshold"], config.review_rate)
np.testing.assert_array_equal(reviewed, holdout.manual_review.to_numpy())
ordered = holdout.sort_values([TIME, ID]).copy()
days = ordered[TIME] // 86400
seen = ordered.groupby(days).cumcount() + 1
reviews_so_far = ordered.manual_review.groupby(days).cumsum()
assert (reviews_so_far <= np.floor(config.review_rate * seen + 1e-12)).all()

daily = pd.read_csv(artifact_dir / "daily_policy.csv")
display(daily.head(10))
fig, ax = plt.subplots()
daily.plot(x="day", y="review_rate", ax=ax, label="Observed review rate")
ax.axhline(config.review_rate, color="#df7248", linestyle="--", label="Capacity limit")
ax.set(title="Holdout Review Allocation", xlabel="Relative day", ylabel="Review rate")
ax.legend()
plt.tight_layout()
plt.show()

## 3. State Continuity Across Batches

ReviewState retains the current day, observed transaction count and allocated reviews.
Chronological batches share this state, making their combined decisions equivalent
to processing the same event stream in one batch.

In [ ]:
state = ReviewState()
cut = len(holdout) // 2
first = online_review(
    holdout.iloc[:cut],
    scores[:cut],
    selection["threshold"],
    config.review_rate,
    state,
)
second = online_review(
    holdout.iloc[cut:],
    scores[cut:],
    selection["threshold"],
    config.review_rate,
    state,
)
np.testing.assert_array_equal(np.r_[first, second], reviewed)
display(pd.Series({"batch_equivalence": True, "prefix_quota_valid": True}))

## 4. Expected Costs and Retrospective Benchmark

Review precision measures fraud concentration among selected transactions; review recall measures the share
of all fraud reaching review. Expected prevention additionally depends on reviewer sensitivity.
Daily top-k uses all scores from a completed day and therefore serves only as a retrospective benchmark.

In [ ]:
retrospective = daily_top_k(holdout, scores, config.review_rate)
cost_args = (config.fraud_loss, config.review_cost, config.review_sensitivity)
comparison = pd.DataFrame(
    {
        "online": business_metrics(labels, reviewed, *cost_args),
        "daily_top_k_retrospective": business_metrics(labels, retrospective, *cost_args),
    }
).T
display(
    comparison[
        [
            "reviews",
            "review_rate",
            "fraud_in_review",
            "review_precision",
            "review_recall",
            "expected_prevented_frauds",
            "no_review_cost",
            "expected_cost",
            "expected_savings",
        ]
    ]
)
sensitivity = pd.read_csv(artifact_dir / "cost_sensitivity.csv")
display(
    sensitivity.pivot(
        index="review_sensitivity",
        columns="fraud_loss",
        values="expected_savings",
    )
)

## 5. Selected-Model Importance and Holdout Drift

Feature importance describes the selected estimator using its own attribution definition. It does not establish
causal effects or support direct magnitude comparisons between model families. Population stability index compares
monitored distributions with calibration references; 0.2 is an investigation heuristic rather than proof of quality.

In [ ]:
importance = pd.read_csv(artifact_dir / "feature_importance.csv")
fig, ax = plt.subplots(figsize=(10, 6))
importance.head(15).iloc[::-1].plot.barh(x="feature", y="importance", ax=ax, legend=False)
ax.set(title=f"{selected_name}: {importance.method.iloc[0]}", xlabel="Importance", ylabel="Feature")
plt.tight_layout()
plt.show()

drift = pd.read_csv(artifact_dir / "drift_holdout.csv")
display(drift.sort_values("psi", ascending=False))

## 6. Unlabeled Inference and Submission Integrity

Batch inference applies the frozen feature mappings, classifier, calibrator and online review policy.
Submission probabilities remain aligned with sample_submission transaction order.
Finite probabilities, unique identifiers and cumulative quota compliance establish output consistency,
without implying measured classification quality on unlabeled data.

In [ ]:
from fraud_detection.inference import predict

predictions = predict(config)
submission = pd.read_csv(artifact_dir / "submission.csv")
sample = pd.read_csv(Path(config.raw_dir) / "sample_submission.csv", usecols=[ID])
assert submission.columns.tolist() == [ID, TARGET]
assert submission[ID].equals(sample[ID])
assert submission[ID].is_unique
assert np.isfinite(submission[TARGET]).all() and submission[TARGET].between(0, 1).all()
assert len(predictions) == len(submission)
ordered_test = predictions.sort_values([TIME, ID])
test_days = ordered_test[TIME] // 86400
test_seen = ordered_test.groupby(test_days).cumcount() + 1
test_reviews = ordered_test.manual_review.groupby(test_days).cumsum()
assert (test_reviews <= np.floor(config.review_rate * test_seen + 1e-12)).all()
display(submission.head())
display(
    pd.Series(
        {
            "test_rows": len(submission),
            "reviews": int(predictions.manual_review.sum()),
            "submission_order_valid": True,
            "test_prefix_quota_valid": True,
        }
    )
)
display(pd.read_csv(artifact_dir / "drift_test.csv").sort_values("psi", ascending=False))

In [ ]:
%matplotlib inline

## 7. Batch Score Consistency

Scoring a small transaction batch with the saved artifact reproduces its corresponding inference probabilities.
Review decisions depend on the full chronological state, so this probability comparison
is separate from a daily-capacity simulation.

In [ ]:
import joblib
import pyarrow.parquet as pq

model = joblib.load(artifact_dir / "model.joblib")
batch = next(pq.ParquetFile(processed_dir / "test.parquet").iter_batches(batch_size=12)).to_pandas()
batch_scores = model.predict_proba(batch)
preview = batch[[ID, TIME, "TransactionAmt"]].copy()
preview["risk_score"] = batch_scores
saved_scores = predictions.set_index(ID).loc[preview[ID], "risk_score"].to_numpy()
np.testing.assert_allclose(batch_scores, saved_scores)
display(preview)

## Operational Interpretation

Review precision, recall and expected savings depend on the actual allocation rate and the assumed reviewer sensitivity.
Financial outcomes remain simulations under the stated costs. Production applicability also depends on
point-in-time feature availability, label delay and review latency. Subsequent labels support renewed
assessment of calibration, fraud recall and realized costs.